# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

In [2]:
# from langchain_community.document_loaders import PyPDFLoader

# #file_path = "../05_src/documents/managing_oneself_drucker.pdf"
# file_path = "../05_src/documents/ai_report_2025.pdf"
# loader = PyPDFLoader(file_path)

# docs = loader.load()

# print(len(docs))

In [3]:
# document_text = ""
# for page in docs:
#     document_text += page.page_content + "\n"

In [4]:
#print(len(document_text)) # managing_oneself_drucker.pdf    51452
#print(len(document_text)) # ai_report_2025.pdf              53851

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.  

The article is not in a url because there is a paywall, Jesus gave us the html file, the article is not a local file.

In [5]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

file_path = "../05_src/documents/what_is_noise_the_new_yorker.htm"
loader = UnstructuredHTMLLoader(file_path)
document = loader.load()

In [6]:
print(type(document))
print(type(document[0]))
print(type(document[0].page_content))
document_text = document[0].page_content

<class 'list'>
<class 'langchain_core.documents.base.Document'>
<class 'str'>


In [7]:
len(document_text) # what_is_noise_the_new_yorker.htm       33872

33872

In [8]:
document_text

'Text\n\nText size\n\nLayout\n\nTheme\n\nRead aloud\n\nVoice\n\nnewyorker.com\n\nWhat Is Noise?\n\nAlex Ross\n\n31–39 minutes\n\nSometimes we embrace it, sometimes we hate it—and everything depends on who is making it.\n\nApril 15, 2024\n\nNoise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra Péterffy\n\n“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din within our minds. The demented narrator of Poe’s “The Tell-Tale Heart” jabbers about noise while he hallucinates his victim’s heartbeat: “I found that the noise was not within my ears. . .

### Cleaning document 

I tried couple of loaders but gave me trouble:

- BSHTMLLoader: the article has a different encoding than the default utf-8, I needed to play with them and look at the result, or to use charder library to detect what specific encoding was used  

- WebBaseLoader: I need to use file:// protocol with the absolute path because the file is local. My computer username has a space in it (many years ago I did not know I was liening towards coding that much :). I played with "%20", with urllib.parse.quote and with pathlib.Path.as_uri(), none worked, I needed to install additional package.

I chose to abort these methodologies in order to focus on the core of the course. 

I used UnstructuredHTMLLoader, which is unstructured and does not extract tags from the document. 

I ended up doing a manual cleaning, aware that will not be reproducible with a different article

In [9]:
import re

# Find text between "newyorker.com" and "The New Yorker Classics Newsletter"
match = re.search(r'newyorker\.com(.*?)The New Yorker Classics Newsletter', document_text, flags=re.DOTALL)

if match:
    cleaned_text = match.group(1).strip()  # .strip() removes leading/trailing whitespace
    # print(cleaned_text)
else:
    print("Pattern not found")

In [10]:
#len(document_text)     # what_is_noise_the_new_yorker.htm       33872
len(cleaned_text)       # what_is_noise_the_new_yorker.htm       31267, removed 33872-31267 = 2605 characters, which is 7.7% of original article

31267

In [11]:
article = cleaned_text

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
from pydantic import BaseModel, Field
from openai import OpenAI
client = OpenAI()

In [13]:
class StructuredOutput(BaseModel):
    author: str=Field(description="The author of the article")
    title: str=Field(description="The title of the article")
    relevance: str=Field(description="Relevance statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development")
    summary: str=Field(description=" Summary of the article concise and succinct")
    tone: str=Field(default="Victorian English", description="Distinguishable tone or style of language used to produce the summary that was required in the prompt; use Victorian English if none required")
    inputtokens: int=Field(description="Input tokens")
    outputtokens: int=Field(description="Output tokens")
    model: str=Field(description="The model used to create this output")

In [14]:
max_output_token = 1000

#### System message

In [15]:
system_prompt = f"""
                You are a journalist specialized in AI (artificial intelligence) publications. 
                Create a summary of the attached article and a relevance statement, both drafted in a distinguishable tone or style of English language indicated in the prompt or if none indicated using the tone by default.
                Create a response or output with less than {max_output_token} tokens, unless a different number of tokens, words, characters or in general different length is required in the promtp, user promt or input.
                If you detect a query that does not contain a petition of summarizing an article or document, return the output "This bot forgot what you did not" in all fields except in the field model.
                When asked factual questions in the prompt, do not make assumptions and do not return examples as a replacement. Follow the instructions verbatim.
                """

Repository of instructions snippets used in interim versions:  

The output or response of any query must always be a Pydantic BaseModel object. : it is redundant because the response from OpenAI we use already is. And this statement introduced extra wording in the response

The instruction: "If model is indicated as model = 'gpt-5', do not use model = 'gpt-5', use model = 'gpt-4o' instead" worked when asking for model = 'gpt-5' replied a query using 'gpt-4o', but it took more than one minute to reply, which makes me assume that it did the query at least twice, first time with 'gpt-5' forced by model=, then it was not acceptable as per the instructions, then it did it again using 'gpt-4o'.  
And, if model required was not 'gpt-5', it always used 'gpt-4o', and I was not able to use 'gpt-4o-mini'. I tried to add "Otherwise, use model indicated by model =.", did not work.

#### Query

In [16]:
prompt = f"""
    Given the following article, produce the specified outputs.
    Use the distinguishable Legalese language, meaning legal language.

    The article is the following: 
    <article>
    {article}
    </article>
"""

Repository of prompt snippets used in interim versions:  
&nbsp;&nbsp;&nbsp;&nbsp;6. Check how many Canadian Dollars (CAD) will you charge the OpenAI account for this query, expressed in $/10,000.00 queries like this one,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Cost: <cost expressed as dollars per 10,000 similar queries (e.g., $45.50/10k)>:  
&nbsp;&nbsp;&nbsp;&nbsp;I tried few variations, numbers were not reliable, it is not required, I removed it  

&nbsp;&nbsp;&nbsp;&nbsp;5. Check the number of input tokens and the number of output tokens used  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;InputTokens: <inputtokens>  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;OutputTokens: <outputtokens>  
&nbsp;&nbsp;&nbsp;&nbsp;It did not return reliable numbers. I switched to the command  

&nbsp;&nbsp;&nbsp;&nbsp;5. Check what model did you actually use to generate this output. Do not make assumptions, actually check
&nbsp;&nbsp;&nbsp;&nbsp;Model: <model>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;I asked for     model = 'gpt-4o-mini', and I got  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Model="gpt-3.5-turbo"  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Model="gpt-4"  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Model="gpt-3.5-turbo"  # Assume this is the model used  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model="gpt-4"        # Example model used  
&nbsp;&nbsp;&nbsp;&nbsp;It did not work well. I better ask the object (response)


When prompted the prompt below, the response was "This bot forgot what you did not", as stated in the system prompt;  it worked.

In [17]:
# prompt = f"""
#         What time is it?
#         """

In [18]:
# Code Archive: the code below also works
# response = client.responses.parse(
#     #model = 'gpt-4o-mini',
#     model = 'gpt-4o',
#     #model = 'gpt-5',
#     instructions = system_prompt,
#     input = prompt,
#     text_format=StructuredOutput,
# )

In [19]:
response = client.responses.parse(
    model = 'gpt-4o',
    input=[{"role": "system", "content": system_prompt}, 
           {"role": "user", "content": prompt}],
    text_format=StructuredOutput,
)

model = 'gpt-4o-mini' and 'gpt-4o' responses, take between 5 and 8 seconds

In [20]:
print(type(response))
print(type(response.output_text))
print(type(response.output_parsed))

<class 'openai.types.responses.parsed_response.ParsedResponse[StructuredOutput]'>
<class 'str'>
<class '__main__.StructuredOutput'>


In [21]:
# Get actual token counts from the response
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
total_tokens = response.usage.total_tokens
model_used = response.model

Presenting response

In [22]:
# response.to_dict()
# response.to_json()

In [23]:
print(f"Author: {response.output_parsed.author}")
print(f"Title: {response.output_parsed.title}")
print(f"Relevance: {response.output_parsed.relevance}")
print(f"Summary: {response.output_parsed.summary}")
print(f"Tone: {response.output_parsed.tone}")
print(f"Input tokens, as per structured output: {response.output_parsed.inputtokens}")
print(f"Output tokens, as per structured output: {response.output_parsed.outputtokens}")
print(f"Model, as per structured output: {response.output_parsed.model}")
print(f"Input tokens, attribute from object response: {input_tokens}")
print(f"Output tokens, attribute from object response: {output_tokens}")
print(f"Total tokens, attribute from object response: {total_tokens}")
print(f"As per attribute from object response, the model used is: {response.model}")

Author: Alex Ross
Title: What Is Noise?
Relevance: This article presents a profound exploration of noise, its definitions, cultural implications, and intersections with music, offering valuable insights for AI professionals working with audio processing, signal analysis, and information theory. Understanding noise and its multifaceted nature enhances the development of algorithms designed to mitigate or utilize noise in various AI applications, from natural language processing to autonomous systems.
Summary: In this comprehensive treatise, Alex Ross examines the concept of 'noise,' tracing its etymological roots and cultural significance across history. He explores how noise intertwines with music, often blurring boundaries and challenging aesthetic norms. From classical compositions to avant-garde music and urban soundscapes, noise serves as both an artistic medium and a societal echo. Ross delves into historical efforts at noise control, the juxtaposition of music and noise, and the 

#### Outputs obtained with different models:

Author: Alex Ross  
Title: What Is Noise?  
Relevance: The article provides an in-depth analysis of the concept of noise, touching on its cultural, historical, and philosophical implications. For AI professionals, understanding the multifaceted nature of noise is crucial, as it has direct implications for areas like natural language processing, audio analysis, and the handling of 'noise' in data sets. This knowledge aids in developing systems that better interpret and manage real-world complexities.  
Summary: The article explores the concept of noise, tracing its origins from nuisance and madness to its potentially majestic and liberating facets. It examines how noise has been perceived across different cultures and languages, and its role in music, particularly in avant-garde and experimental genres. The discussion extends to noise's impact on society, its historical regulation attempts, and modern technological and informational noise. The author also shares personal experiences with noise, illustrating its subjective nature. Ultimately, noise is portrayed as both a medium of control and a form of resistance.  
Tone: Legalese  
Input tokens, as per structured output: 876  
Output tokens, as per structured output: 215  
Model, as per structured output: GPT-4  
Input tokens, attribute from object response: 7180  
Output tokens, attribute from object response: 223  
Total tokens, attribute from object response: 7403  
As per attribute from object response, the model used is: gpt-4o-2024-08-06  

Author: Alex Ross  
Title: What Is Noise?  
Relevance: The exploration of noise within this article is highly pertinent to AI professionals, as it emphasizes the intricate balance between data, information, and the ambient 'noise' that may distort or enhance both human and machine learning processes. Mastery over the conceptualization of noise may foster improved data handling and algorithmic design in AI systems.  
Summary: The article discusses the multifaceted nature of noise, encompassing its historical, cultural, and psychological dimensions. It delineates noise as both a source of distress and a means of artistic expression, revealing its power dynamics within society. The discourse extends from personal experiences of noise to its implications in public life and technological advancements, ultimately advocating for a nuanced understanding of how noise shapes human experience.  
Tone: Formal and analytical, reflective of an academic discourse on a nuanced topic.  
Input tokens, as per structured output: 139  
Output tokens, as per structured output: 703  
Model, as per structured output: gpt-4  
Input tokens, attribute from object response: 7180  
Output tokens, attribute from object response: 191  
Total tokens, attribute from object response: 7371  
As per attribute from object response, the model used is: gpt-4o-mini-2024-07-18  

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
